# n01 — Sector 識別性検証: chunks 抽出 + embedding 生成

**ActionItem**: `act-2026-05-29-005` / **Decision**: `dec-2026-05-29-005`, `dec-2026-05-29-008`

`data/processed/sector_validation/ticker_list.csv`（11 GICS sector × 5 ticker = 55 銘柄,
`act-2026-05-29-004` 成果物）を入力に、`fiscal_year >= 2020` × `form in (10-K, 10-Q)` ×
全 4 section（item_1 / item_1a / item_7 / item_2）の chunks を抽出し、
`gte-Qwen2-1.5B-instruct`（MPS bfloat16）で 1536 次元 embedding を生成する。
中間ファイルを NAS に保存し、`n02_analyze_and_topics.ipynb` から疎結合に参照する。

## 設計方針

- **embedding ロジックは再実装しない**。HF1 確定済みの
  `notebook/FILING_NLP/pipeline/embed_indices.py` の関数を import して再利用する:
  `_setup_hf_cache` / `_load_model` / `encode_texts` / `last_token_pool`。
  これらは last_token_pool + L2 正規化 + `use_cache=False` を内包し、
  (N, 1536) float32（L2 正規化済み）を返す。
- **抽出ロジック**は `sector_validation/extract_chunks.py` に切り出して再利用する
  （notebook と CLI 双方から呼べる）。
- **combined 単一配列 + NaN-marker resume**: `embeddings.npy` を
  `np.full((N, 1536), np.nan, float32)` で初期化し、`np.isnan(arr).any(axis=1)` で
  未処理 index を取得。`CHECKPOINT_EVERY_N_BATCHES` ごとに `np.save` で永続化。
- **行順の厳密一致**: `chunks_meta` を `reset_index(drop=True)` で確定し、
  `embeddings.npy` の行 i ↔ `chunks_meta` 行 i を保証する。
- 出力先: NAS `embeddings/sector_validation/{embeddings.npy, chunks_meta.parquet}`。

> **transformers v5 互換注記（2026-05-29 修正, `dec-2026-05-29-008`）**: gte-Qwen2 は
> GTE 設計で **bidirectional attention**（causal LLM の attention mask を外して双方向
> エンコーダ化）で埋め込む。`modeling_qwen.py` の `Qwen2Model.forward` は
> `is_causal` デフォルトが `False`（sdpa で `_prepare_4d_attention_mask_for_sdpa` =
> 双方向）であり、標準 `transformers` の `Qwen2Model`（常時 causal）に置換すると
> 埋め込みが**別物**になる（pilot `dec-2026-05-22-emb` も bidirectional 生成）。
> transformers 5.1.0 では HF Hub カスタムコードが 2 点で壊れるが、bidirectional は
> 本質のため次の対処でカスタムコードを維持する（`embed_indices._load_model`）:
> (1) tokenizer: 削除済み `Qwen2TokenizerFast` 継承を回避し標準 slow `Qwen2Tokenizer`
> を使用（vocab 同一、pilot100 で encode 一致）。
> (2) model: v5 で `rope_parameters` dict に移動した `config.rope_theta`
> （= 1000000.0）を config に復元注入し、`trust_remote_code=True` でカスタム
> bidirectional encoder をロード（load 成功・`module=modeling_qwen`・L2 norm≈1.0・
> NaN 0 を検証済み）。
>
> **実測スループット (MPS bfloat16, batch=32, max_length=512, bidirectional)**:
> 0.729 chunks/sec（約 44 s/batch）。本検証サンプル 107,984 chunks のフル生成は
> 約 41.2 h と推定。`padding=True` でバッチ内最長 chunk に合わせるため、
> `token_count` 順ソートで padding 無駄を削減すれば短縮余地がある（将来最適化）。


In [ ]:
# Cell 2: imports + HF cache 設定 + パラメータ定義
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# repo root を sys.path へ（embed_indices / extract_chunks / utils_core を import 可能に）
REPO_ROOT = Path("/Users/yuki/Desktop/quants")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# embed_indices を import すると import 時に repo root が sys.path へ挿入される。
from notebook.FILING_NLP.pipeline import config, embed_indices
from notebook.FILING_NLP.sector_validation import extract_chunks as ec
from utils_core.logging import get_logger

logger = get_logger("n01_extract_and_embed")

# HF cache を notebook/FILING_NLP/data/hf_cache に向ける（transformers import 前に呼ぶ）。
# PYTORCH_ENABLE_MPS_FALLBACK=1 もここで設定される。
HF_CACHE_DIR = embed_indices._setup_hf_cache()
logger.info("HF cache configured", hf_home=os.environ.get("HF_HOME"))

# === パラメータ（config 定数を一次情報とし、ハードコードしない） ===
BATCH_SIZE = 32  # dec-2026-05-29-005 指定。MPS OOM 時は 16 にフォールバック（下記注記）
MAX_LENGTH = config.EMBEDDING_MAX_LENGTH  # 512
DEVICE = config.EMBEDDING_DEVICE  # "mps"
DTYPE = config.EMBEDDING_DTYPE  # "bfloat16"
FISCAL_YEAR_MIN = ec.FISCAL_YEAR_MIN  # 2020
FORMS = ec.FORMS  # ("10-K", "10-Q")
CHECKPOINT_EVERY_N_BATCHES = 20

# 入出力パス
TICKER_LIST_CSV = (
    REPO_ROOT / "data" / "processed" / "sector_validation" / "ticker_list.csv"
)
CHUNKS_DIR = config.CHUNKS_DIR / "indices_v1"
OUT_DIR = config.EMBEDDINGS_DIR / "sector_validation"
OUT_EMB = OUT_DIR / "embeddings.npy"
OUT_META = OUT_DIR / "chunks_meta.parquet"

# 環境スイッチ: N01_SMOKE=1 で小実測のみ実行（フル生成セルは skip）
SMOKE = os.environ.get("N01_SMOKE", "0") == "1"
SMOKE_N = 320

print(f"BATCH_SIZE={BATCH_SIZE} MAX_LENGTH={MAX_LENGTH} DEVICE={DEVICE} DTYPE={DTYPE}")
print(
    f"FISCAL_YEAR_MIN={FISCAL_YEAR_MIN} FORMS={FORMS} CHECKPOINT_EVERY_N_BATCHES={CHECKPOINT_EVERY_N_BATCHES}"
)
print(f"OUT_DIR={OUT_DIR}")
print(f"SMOKE={SMOKE} (N01_SMOKE env)")

In [ ]:
# Cell 3: chunks 抽出（55 CIK × fiscal_year>=2020 × form in (10-K,10-Q) × 全 section）
#
# extract_chunks.extract_chunks が以下を実施する:
#   - 各 CIK の chunks_cik{cik:010d}.parquet を読み、fiscal_year/form でフィルタ
#   - GICS sector/industry を cik で left join（GICS 列名のまま付与）
#   - pd.concat → reset_index(drop=True) で行順を確定
ticker_list = ec.load_ticker_list(TICKER_LIST_CSV)
result = ec.extract_chunks(
    ticker_list, CHUNKS_DIR, fiscal_year_min=FISCAL_YEAR_MIN, forms=FORMS
)
chunks_meta = result.chunks_meta

OUT_DIR.mkdir(parents=True, exist_ok=True)
chunks_meta.to_parquet(OUT_META, index=False)
logger.info("chunks_meta saved", path=str(OUT_META), rows=len(chunks_meta))

# === 抽出統計 ===
print(f"=== 総 chunks 数: {len(chunks_meta):,} ===\n")
print("--- sector別 chunks 数 ---")
print(chunks_meta.groupby("sector").size().sort_values(ascending=False).to_string())
print(f"n_sectors: {chunks_meta['sector'].nunique()}\n")

print("--- form別 ---")
print(chunks_meta.groupby("form").size().to_string())
print()

print("--- fiscal_year別 ---")
print(chunks_meta.groupby("fiscal_year").size().to_string())
print()

print("--- section_key別 ---")
print(chunks_meta.groupby("section_key").size().to_string())
print()

tc = chunks_meta["token_count"]
print("--- token_count 統計 ---")
print(f"mean={tc.mean():.1f} median={tc.median():.1f} max={tc.max()} min={tc.min()}")
print()

n_with = sum(1 for v in result.per_cik_counts.values() if v > 0)
print(f"--- ticker coverage: {n_with}/{len(ticker_list)} ticker に chunks あり ---")
if result.tickers_without_chunks:
    print("chunks が無い ticker:", result.tickers_without_chunks)
else:
    print("全 ticker に chunks あり")
print(f"\n保存先: {OUT_META}")
print(f"chunks_meta: rows={len(chunks_meta)} cols={list(chunks_meta.columns)}")

## エンコード方針

- `embed_indices.encode_texts` を再利用する（バッチ encode → `last_token_pool` →
  `F.normalize(p=2, dim=1)` → float32 numpy、`use_cache=False`）。
- **combined 単一配列 + NaN-marker resume**:
  - `OUT_EMB` が存在し shape==(N, 1536) かつ float32 なら load、
    `np.isnan(arr).any(axis=1)` を未処理 index とする。
  - なければ `np.full((N, 1536), np.nan, np.float32)` で初期化。
- `CHECKPOINT_EVERY_N_BATCHES` バッチごとに `np.save(OUT_EMB, arr)` で永続化。
- **`SMOKE=True` のとき**: 未処理先頭 `min(N, 320)` chunk だけ encode して時間を計測し、
  `chunks/sec` と推定フルランタイムを print して **`OUT_EMB` には保存せず終了**
  （measurement 専用。フル生成は親エージェントが別途 background 起動する）。


In [ ]:
# Cell 5: 全件エンコード（NaN-marker resume 対応）/ SMOKE 時は小実測のみ
N = len(chunks_meta)
dim = config.EMBEDDING_VECTOR_DIM  # 1536
texts_all: list[str] = chunks_meta["text"].tolist()


def _load_or_init_embeddings(out_emb, n: int, d: int) -> np.ndarray:
    """既存 .npy があれば resume 用に load、なければ NaN 初期化配列を返す.

    Parameters
    ----------
    out_emb : Path
        embeddings.npy のパス。
    n : int
        全 chunks 数（chunks_meta の行数）。
    d : int
        embedding 次元（1536）。

    Returns
    -------
    numpy.ndarray
        shape (n, d), float32。未処理行は NaN。
    """
    if out_emb.exists():
        arr = np.load(out_emb, allow_pickle=False)
        if arr.shape == (n, d) and arr.dtype == np.float32:
            done = int((~np.isnan(arr).any(axis=1)).sum())
            logger.info("resume from existing embeddings", done=done, total=n)
            return arr
        logger.warning(
            "existing embeddings shape/dtype mismatch; re-init",
            shape=str(arr.shape),
            dtype=str(arr.dtype),
        )
    return np.full((n, d), np.nan, dtype=np.float32)


# モデルロード（HF cache 未取得なら初回 DL ~3GB が走る）
logger.info(
    "loading model", model_id=config.TOKENIZER_MODEL_ID, device=DEVICE, dtype=DTYPE
)
t_load0 = time.perf_counter()
model, tokenizer = embed_indices._load_model(config.TOKENIZER_MODEL_ID, DEVICE, DTYPE)
print(
    f"model loaded: {type(model).__name__} / tokenizer: {type(tokenizer).__name__} "
    f"({time.perf_counter() - t_load0:.1f}s, 初回は DL 込み)"
)

if SMOKE:
    # === 小実測のみ（OUT_EMB には保存しない） ===
    arr = _load_or_init_embeddings(OUT_EMB, N, dim)
    unprocessed_idx = np.where(np.isnan(arr).any(axis=1))[0]
    smoke_idx = unprocessed_idx[: min(len(unprocessed_idx), SMOKE_N)]
    smoke_texts = [texts_all[i] for i in smoke_idx]
    # ウォームアップ（コンパイル/初回オーバーヘッドを計測から除外）
    _ = embed_indices.encode_texts(
        smoke_texts[:16], model, tokenizer, BATCH_SIZE, MAX_LENGTH
    )
    t0 = time.perf_counter()
    vecs = embed_indices.encode_texts(
        smoke_texts, model, tokenizer, BATCH_SIZE, MAX_LENGTH
    )
    elapsed = time.perf_counter() - t0
    norms = np.linalg.norm(vecs, axis=1)
    cps = len(smoke_texts) / elapsed
    print("=== SMOKE 実測（OUT_EMB 未保存） ===")
    print(f"encode {len(smoke_texts)} chunk: {elapsed:.2f}s  ->  {cps:.3f} chunks/sec")
    print(f"shape={vecs.shape} dtype={vecs.dtype} NaN={int(np.isnan(vecs).sum())}")
    print(
        f"L2 norm mean={norms.mean():.6f} min={norms.min():.6f} max={norms.max():.6f}"
    )
    print(
        f"推定フルランタイム ({N:,} chunks): {N / cps / 60:.1f} min = {N / cps / 3600:.2f} h"
    )
else:
    # === フル生成（resume 対応） ===
    arr = _load_or_init_embeddings(OUT_EMB, N, dim)
    unprocessed_idx = np.where(np.isnan(arr).any(axis=1))[0]
    logger.info("full encode start", unprocessed=len(unprocessed_idx), total=N)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    batches_since_ckpt = 0
    n_batches = int(np.ceil(len(unprocessed_idx) / BATCH_SIZE))
    for b in tqdm(range(n_batches), desc="encode"):
        batch_idx = unprocessed_idx[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        batch_texts = [texts_all[i] for i in batch_idx]
        vecs = embed_indices.encode_texts(
            batch_texts, model, tokenizer, BATCH_SIZE, MAX_LENGTH
        )
        arr[batch_idx] = vecs
        batches_since_ckpt += 1
        if batches_since_ckpt >= CHECKPOINT_EVERY_N_BATCHES:
            np.save(OUT_EMB, arr)
            batches_since_ckpt = 0
    np.save(OUT_EMB, arr)
    logger.info("full encode done", saved=str(OUT_EMB))
    print(f"embeddings saved: {OUT_EMB} shape={arr.shape}")

In [ ]:
# Cell 6: 健全性チェック（SMOKE 時は OUT_EMB 未保存なので skip）
if SMOKE:
    print("SMOKE モードのため OUT_EMB は未保存。健全性チェックを skip します。")
else:
    emb = np.load(OUT_EMB, allow_pickle=False)
    norms = np.linalg.norm(emb, axis=1)
    n_nan_rows = int(np.isnan(emb).any(axis=1).sum())
    print(
        f"shape: {emb.shape}  (期待: ({len(chunks_meta)}, {config.EMBEDDING_VECTOR_DIM}))"
    )
    print(f"dtype: {emb.dtype}")
    print(f"NaN 行数: {n_nan_rows}")
    print(
        f"L2 norm: mean={norms[~np.isnan(norms)].mean():.6f} "
        f"min={np.nanmin(norms):.6f} max={np.nanmax(norms):.6f}"
    )

    assert emb.shape == (len(chunks_meta), config.EMBEDDING_VECTOR_DIM), "shape 不一致"
    assert emb.dtype == np.float32, "dtype が float32 でない"
    assert len(chunks_meta) == emb.shape[0], "chunks_meta と embeddings の行数不一致"
    assert n_nan_rows == 0, f"未処理 (NaN) 行が {n_nan_rows} 件残っている"
    print(
        "\n全チェック通過: embeddings.npy と chunks_meta.parquet は行整合・正規化済み。"
    )